# 02 — Pré-processamento



In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

RANDOM_STATE = 42 #semente fixa para usar em todo o modelo
RAW_FILE1 = Path("..") / "data" / "raw" / "application_record.csv"
RAW_FILE2 = Path("..") / "data" / "raw" / "credit_record.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "TARGET"                                            # PREENCHER: variável alvo

pd.set_option("display.max_columns", None)

# Juntando as bases de dados

In [28]:
#Carregando os dados
dados_pessoais = pd.read_csv(RAW_FILE1)
historico_credito = pd.read_csv(RAW_FILE2)

#criacao de novos dataframes para não modificar a base de dados original.
df_cad = dados_pessoais.copy()
df_cred = historico_credito.copy()

#remapeando a variável STATUS de modo que seus valores sejam apenas números
map_status = {'X':-2,'C':-1, '0':0, '1':1, '2':2, '3':3, '4':4, '5':5}
df_cred['STATUS'] = df_cred['STATUS'].map(map_status)

df_cred = df_cred.groupby('ID').agg(
    WORST_STATUS = ('STATUS','max'), # cria uma coluna que guarda a pior faixa de atraso
    MONTHS_OBS = ('MONTHS_BALANCE','count'),#cria uma coluna c/ a quantidade de meses observados
    MONTH_BEGAN = ('MONTHS_BALANCE','min')# cria uma coluna que informa há quantos meses o cliente é observado
).reset_index()

df_merge = pd.merge(df_cad,df_cred,how='inner',on='ID')

In [29]:
df_merge

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,WORST_STATUS,MONTHS_OBS,MONTH_BEGAN
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,16,-15
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,15,-14
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0,30,-29
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0,5,-4
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,-2,5,-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,1,0,0,0,Managers,2.0,5,12,-11
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,5,24,-23
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,5,33,-32
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,1,0,0,0,Sales staff,2.0,2,10,-9


A utilização do `inner` é essencial, pois seleciona apenas os clientes que possuem histórico de crédito, crucial para o treinamento e teste do modelo.

## 1. Dados faltantes

In [30]:
nulos = df_merge.isna().sum()
pd.DataFrame({"nulos": nulos,
              "pct": (nulos / len(df_merge) * 100).round(2)}).query("nulos > 0")

,nulos,pct
OCCUPATION_TYPE,11323,31.06


Conforme já explorado no notebook **01_eda**, tais linhas possuem dados preciosos que não podem ser apenas removidos.

In [31]:
df_nulos = df_merge[df_merge['OCCUPATION_TYPE'].isna()] #criação de um dataframe apenas com as linhas que possuem valores nulos


In [32]:
qtd_sem_ocupacao = df_nulos['NAME_INCOME_TYPE'].value_counts().rename('Qtd Sem ocupação') #cria uma serie que conta a ocorrencia por categoria da coluna do dataframe de nulos
qtd_total = df_merge['NAME_INCOME_TYPE'].value_counts().rename('Qtd Total') #cria uma serie que conta a ocorrencia por categoria da coluna do dataframe original
#criação de um dataframe formado pelas duas series
df_resumo_ocupacao = pd.merge(qtd_sem_ocupacao,qtd_total,left_index=True, right_index=True)
df_resumo_ocupacao['Pct Sem Ocupação %'] = round((df_resumo_ocupacao['Qtd Sem ocupação']/df_resumo_ocupacao['Qtd Total']*100),2)
df_resumo_ocupacao

,Qtd Sem ocupação,Qtd Total,Pct Sem Ocupação %
NAME_INCOME_TYPE,,,
Pensioner,6139,6152,99.79
Working,3197,18819,16.99
Commercial associate,1438,8490,16.94
State servant,548,2985,18.36
Student,1,11,9.09


In [33]:
#distribuicao de frequencia das categorias por quantidade de linhas nulas da coluna OCCUPATION_TYPE
(df_nulos['NAME_INCOME_TYPE'].value_counts())/df_nulos['NAME_INCOME_TYPE'].value_counts().sum()*100

NAME_INCOME_TYPE
Pensioner               54.217080
Working                 28.234567
Commercial associate    12.699815
State servant            4.839707
Student                  0.008832
Name: count, dtype: float64

Os **Pensioners** (pensionistas) representam 54% das linhas nulas e possuem 99% de suas linhas na coluna "tipo de ocupação" vazias. Além disso, conforme o notebook `01_eda`, todos esses aposentados deste rol de linhas vazias possuem a coluna `DAYS_EMPLOYED` = 365243. Diante disso, a escolha adotada será:

* Manter todas as linhas, pois representam um terço dos dados;
* Preencher as linhas nulas da coluna `OCCUPATION_TYPE` com o valor `"Nao_informado"`;

## 1.1 Perfis Duplicados

In [34]:
no_id = dados_pessoais.copy()
no_id = no_id.drop(columns = 'ID') #criando um dataframe sem a coluna ID

perfil_duplicado = df_merge.duplicated(no_id.columns).sum() #quantidade de linhas com perfil duplicado
perfis_distintos = df_merge.drop_duplicates(subset=no_id.columns).shape[0] #linhas após remoção dos duplicados

print(f"Linhas com perfil idêntico a outra: {perfil_duplicado} ({perfil_duplicado/len(df_merge)*100:.1f}%)")
print(f"Perfis distinto: {perfis_distintos}")


Linhas com perfil idêntico a outra: 26729 (73.3%)
Perfis distinto: 9728


Como visto no notebook **01_eda**, dos 36.457 clientes, 26.729 (73,3%) possuem perfil idêntico ao de outro cliente, restando apenas 9.728 perfis distintos. Em uma divisão aleatória, cópias de um mesmo perfil podem cair no treino e no teste, inflando as métricas por memorização. Além disso, perfis iguais podem ter resultados diferentes, o que cria ruído que limita a capacidade de previsão do modelo.

A decisão adotada é **manter todas as 36.457 linhas** e, em vez de apagar as cópias, criar o identificador `PROFILE_ID`: clientes com cadastro idêntico recebem o mesmo valor.

* `PROFILE_ID` **não é uma feature**. Ele serve apenas para o notebook `03_modelagem` fazer o **split por grupo** (`StratifiedGroupKFold`), garantindo que todos os clientes de um mesmo perfil fiquem do mesmo lado (treino ou teste), sem vazamento por cópia.
* Manter as linhas preserva informação: os rótulos diferentes dentro de um mesmo perfil refletem a variabilidade real do comportamento, e o modelo aprende com 4 vezes mais registros.

In [35]:
# Identificador de perfil: clientes com TODAS as colunas do cadastro (exceto ID) iguais recebem o mesmo PROFILE_ID.
# dropna=False: OCCUPATION_TYPE tem nulos e eles também precisam formar grupo (por padrão o groupby os descartaria).
#ngroup: atribui um numero inteiro sequencial para cada grupo distinto encontrado. Linhas com as mesmas informações cadastrais ficam no mesmo grupo.
df_merge["PROFILE_ID"] = df_merge.groupby(list(no_id.columns), dropna=False).ngroup()

#assert: verifica se a quantidade de valores unicos é igual a da variavel perfis_distintos
assert df_merge["PROFILE_ID"].nunique() == perfis_distintos   # consistente com a contagem acima
print(f"Linhas mantidas: {len(df_merge)}")
print(f"Perfis distintos (grupos): {df_merge['PROFILE_ID'].nunique()}")
df_merge["PROFILE_ID"].value_counts().describe().round(1)

Linhas mantidas: 36457
Perfis distintos (grupos): 9728


count    9728.0
mean        3.7
std         3.4
min         1.0
25%         1.0
50%         3.0
75%         5.0
max        51.0
Name: count, dtype: float64

## 2. Definição da variável alvo

| STATUS ||
| :--- | :--- |
 0: 1-29 dias de atraso <br> 1: 30-59 dias de atraso <br> 2: 60-89 dias de atraso <br> 3: 90-119 dias de atraso <br> 4: 120-149 dias de atraso <br> 5: Dívida vencida, em cobrança ou baixada há mais de 150 dias <br> C: Quitado no mês <br> X: Sem empréstimo no mês |


In [36]:
range_atraso = [1,2,3,4,5]
pd.DataFrame(
    {"Critério": f"Pior Status >= {k}",
     "Quantidade":(df_merge['WORST_STATUS']>=k).sum(),
     "Porcentagem": f"{round((df_merge['WORST_STATUS']>=k).mean()*100,2)} %"}
    for k in range_atraso
    )

,Critério,Quantidade,Porcentagem
0,Pior Status >= 1,4291,11.77 %
1,Pior Status >= 2,616,1.69 %
2,Pior Status >= 3,302,0.83 %
3,Pior Status >= 4,226,0.62 %
4,Pior Status >= 5,180,0.49 %


Apesar de o corte de 30+ dias suavizar o desbalanceamento, optamos por manter o critério tradicional de mercado de 60+ dias para a definição do target. Essa abordagem evita o risco de falso-positivo (falsos maus pagadores), preservando na classe de bons pagadores os clientes com atrasos curtos ou pontuais.

In [37]:
#criação da variável alvo
df_merge['TARGET'] = (df_merge['WORST_STATUS']>=2).astype(int)

In [38]:
#proporção entre bons pagoadores: 0 e maus pagadores: 1, em porcentagem
df_merge['TARGET'].value_counts(normalize=True)*100

TARGET
0    98.310338
1     1.689662
Name: proportion, dtype: float64

A **binarização** é essencial, pois a coluna `WORST_STATUS`, usada para criar a variável alvo, possuía 8 categorais, sendo que 4 delas (2 a 5) somam apenas **1,69%** dos clientes. Tentar prever individualmente cada um dos 8 níveis exigiria um algoritmo de classificação multiclasse. Como algumas classes têm um volume insignificante de exemplos, o modelo teria um desempenho péssimo ao tentar prever esses níveis específicos, além de aumentar desnecessariamente a complexidade computacional.

Portanto, transformar essa variável em uma resposta binária ($0 =$ "Bom/Aprovar", $1 =$ "Mau/Negar") alinha perfeitamente a modelagem estatística com o objetivo estratégico da empresa.

## 3. Normalização / padronização


* **Primeiro, vamos analisar as escalas das variáveis numéricas.**

In [39]:
num_cols = ["AGE_YEARS", "YEARS_EMPLOYED", "AMT_INCOME_TOTAL",
            "CNT_CHILDREN", "CNT_FAM_MEMBERS", "MONTHS_OBS"]
#criando um novo dataframe com novas colunas de data convertidas para anos
df_escala = df_merge.assign(
    AGE_YEARS = (-df_merge['DAYS_BIRTH']/365.25).astype(int),
    YEARS_EMPLOYED = np.where(df_merge['DAYS_EMPLOYED']>0,0,(-df_merge['DAYS_EMPLOYED']/365.25).astype(int))
)

df_escala[num_cols].agg(["min", "max", "mean", "std"]).T.round(2)

,min,max,mean,std
AGE_YEARS,20.0,68.0,43.23,11.50
YEARS_EMPLOYED,0.0,43.0,5.61,6.41
AMT_INCOME_TOTAL,27000.0,1575000.0,186685.74,101789.23
CNT_CHILDREN,0.0,19.0,0.43,0.74
CNT_FAM_MEMBERS,1.0,20.0,2.20,0.91
MONTHS_OBS,1.0,61.0,21.33,14.91


**Por que padronizar?** É necessário porque as **escala são incompatíveis**. A renda anual vai de 27.000 a 1.575.000, enquanto o número de filhos de 0 a 19 - diferença de 5 ordem de grandeza. Sem padronização, regressão logística regularizada penaliza desproporcionalmente os coeficientes das variáveis de escala pequena, e KNN/SVM têm a distância inteiramente dominada pela renda.

Optou-se pelo `StandardScaler` em razão da alta disparidade na renda (máximo de R$ 1,575 mi vs. mediana de R$ 157,5 mil). O uso do `MinMaxScaler` esmagaria 95% dos registros no intervalo de 0 a 0,25, destruindo a variabilidade útil da amostra. O `StandardScaler`, por se fundamentar em média e desvio-padrão, preserva a dispersão dos dados e é mais robusto a esse tipo de distorção.

Abaixo uma prévia de como ficarão dados normalizados pelo **StandardScaler**:


In [40]:
demo = StandardScaler().fit_transform(df_escala[num_cols])

pd.DataFrame(demo, columns=num_cols).agg(["mean", "std", "min", "max"]).T.round(2)

,mean,std,min,max
AGE_YEARS,0.0,1.0,-2.02,2.15
YEARS_EMPLOYED,-0.0,1.0,-0.88,5.83
AMT_INCOME_TOTAL,0.0,1.0,-1.57,13.64
CNT_CHILDREN,-0.0,1.0,-0.58,25.01
CNT_FAM_MEMBERS,0.0,1.0,-1.31,19.53
MONTHS_OBS,0.0,1.0,-1.36,2.66


## 4. Feature engineering


In [41]:
df_feature = df_merge.copy()

In [42]:
df_feature

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,WORST_STATUS,MONTHS_OBS,MONTH_BEGAN,PROFILE_ID,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,16,-15,9229,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,15,-14,9229,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0,30,-29,8495,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0,5,-4,3726,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,-2,5,-26,3726,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,1,0,0,0,Managers,2.0,5,12,-11,9159,1
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,5,24,-23,2802,1
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,5,33,-32,2847,1
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,1,0,0,0,Sales staff,2.0,2,10,-9,3781,1


1. Trabalhar as colunas ligadas a "ocupação", criando o marcador "sem ocupação".

In [43]:
df_feature["OCCUPATION_TYPE"] = df_feature["OCCUPATION_TYPE"].fillna("nao_informado") #preenche as linhas nulas com o valor "nao_informado"


2. Conversão das unidades de data

In [44]:
df_feature['AGE_YEARS'] = (-df_feature['DAYS_BIRTH']/365.25).astype(int)


df_feature['YEARS_EMPLOYED'] = np.where(df_feature['DAYS_EMPLOYED'] > 0, 0, (-df_feature['DAYS_EMPLOYED'])/365.25).astype(int)

3. Winsorização da renda no percentil 99 e criação da renda per capita

In [45]:
p99 = df_feature["AMT_INCOME_TOTAL"].quantile(0.99)
df_feature["AMT_INCOME_TOTAL"] = df_feature["AMT_INCOME_TOTAL"].clip(upper=p99)

df_feature["PER_CAPITA_INCOME"] = (df_feature["AMT_INCOME_TOTAL"] / df_feature["CNT_FAM_MEMBERS"]).round(2)


4. Resolução da multicolinearidade (CNT_CHILDREN vs. CNT_FAM_MEMBERS)

Conforme identificado no notebook 01_eda, existe uma forte correlação linear (0,89) entre o número de filhos e o total de membros da família. Para eliminar essa redundância, a variável de membros da família foi desmembrada em duas: quantidade de adultos e quantidade de filhos.

In [46]:
df_feature["CNT_ADULTOS"] = (df_feature["CNT_FAM_MEMBERS"] - df_feature["CNT_CHILDREN"]).astype(int)

In [47]:
# CNT_CHILDREN: valores 14 e 19 são erro de cadastro; capados em 5 (máximo plausível observado de forma consistente na distribuição).
df_feature["CNT_CHILDREN"] = df_feature["CNT_CHILDREN"].clip(upper=5)

5. Contatos declarados: Calcula a quantidade de canais de contato informados no cadastro a partir das flags de telefone comercial, telefone e e-mail.

In [48]:
df_feature["CNT_CONTACT"] = df_feature[["FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL"]].sum(axis=1)

6. Remoção das colunas:
* "WORST_STATUS", "MONTHS_OBS", "MONTH_BEGAN", pois foram usadas para construir a variável `TARGET`. Mantê-las causaria **vazamento de dados**.
* "FLAG_MOBIL": constante (=1 para todos) — variância zero, informação zero;
* "DAYS_BIRTH":       substituída por AGE_YEARS,
* "DAYS_EMPLOYED":    substituída por YEARS_EMPLOYED;
* "CNT_FAM_MEMBERS":  colinear com CNT_CHILDREN (0,89) — substituída por CNT_ADULTOS",

In [49]:
remover = [
    "WORST_STATUS",
    "MONTHS_OBS",
    "MONTH_BEGAN",
    "FLAG_MOBIL",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_FAM_MEMBERS",
]
df_final = df_feature.copy()
df_final = df_final.drop(columns = remover)
print("Shape final:", df_final.shape)
print(f"Taxa da classe positiva: {df_final['TARGET'].mean() * 100:.2f}%")
df_final.dtypes.to_frame("tipo")

Shape final: (36457, 21)
Taxa da classe positiva: 1.69%


,tipo
ID,int64
CODE_GENDER,str
FLAG_OWN_CAR,str
FLAG_OWN_REALTY,str
CNT_CHILDREN,int64
AMT_INCOME_TOTAL,float64
NAME_INCOME_TYPE,str
NAME_EDUCATION_TYPE,str
NAME_FAMILY_STATUS,str
NAME_HOUSING_TYPE,str


In [50]:
df_final

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,PROFILE_ID,TARGET,AGE_YEARS,YEARS_EMPLOYED,PER_CAPITA_INCOME,CNT_ADULTOS,CNT_CONTACT
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,0,0,nao_informado,9229,0,32,12,213750.0,2,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,0,0,nao_informado,9229,0,32,12,213750.0,2,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Security staff,8495,0,58,3,56250.0,2,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,3726,0,52,8,270000.0,1,2
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,3726,0,52,8,270000.0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Managers,9159,1,47,6,157500.0,2,0
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,0,1,1,Medicine staff,2802,1,33,3,78750.0,2,2
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,0,1,1,Medicine staff,2847,1,33,3,78750.0,2,2
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Sales staff,3781,1,49,1,141750.0,2,0


## 5. Salvar dataset tratado

In [51]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df_final.to_csv(PROCESSED / "dataset_tratado.csv", index=False)